In [0]:
# 02_bronze – Rohdaten 1:1 als Delta-Tabellen laden (alles als String)
from pyspark.sql import functions as F

RAW = "/Volumes/workspace/banking/raw"
TARGET = "workspace.banking"

def ingest(folder: str, table: str):
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)        # Bronze = unverändert, keine Typ-Interpretation
          .csv(f"{RAW}/{folder}/")
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingest_ts", F.current_timestamp()))
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{TARGET}.bronze_{table}"))
    print(f"bronze_{table}: {df.count()} Zeilen")

ingest("customers", "customers")
ingest("accounts", "accounts")
ingest("transactions", "transactions")